In [6]:
## Packages to be installed- ansys_fluent_core, bayesian_optimization

import ansys.fluent.core as pyfluent
import os
import bayes_opt
from bayes_opt import BayesianOptimization, UtilityFunction

In [7]:
## Function to run each ANSYS FLUENT Simulation
def run_ansys_sim(lmbda,power,eta_0,eta_inf):
    
    global count
    
    ## eta_inf chosen by optimizer always has to be less than eta_0
    if eta_inf > eta_0:
        return 0
    
    ## Can change fluent version number, number of processesors and whether to show GUI or not
    session = pyfluent.launch_fluent("22.2.0",precision="double",version="2d", processor_count=4, mode="solver", show_gui=True)
    tui = session.tui
    tui.file.read_case("RacetrackingLowerPermeability.cas.h5") ## These would change depending 
    tui.file.read_data("RacetrackingLowerPermeability.dat.h5") ## on simulation Case file used

    ## Stores log for each simulation, can use it to check for error and fill time. 
    filename="Simulation_Log"+str(count)
    tui.file.start_transcript(filename)

    ## Changes the carreau model to incorporate new set of parameters
    tui.define.materials.change_create("resin","resin","no","no","no","yes","carreau","shear-rate-dependent",lmbda,power,eta_0,eta_inf)
    
    ## If eta_0 goes below 0.1, we switch to a more accurate time step size of 0.001 s 
    if eta_0<=0.1:
        tui.solve.initialize.initialize_flow()
        tui.solve.set.transient_controls.time_step_size(0.001)
        tui.solve.dual_time_iterate(600000,50,"no","yes")
    else:
        ## In all other cases, we stick with a time-step size of 0.1 s
        tui.solve.initialize.initialize_flow()
        tui.solve.dual_time_iterate(6000,50,"no","yes")
    

    # Checks for Floating Point Error/ Divergence in Simulation log and alters URF accordingly to re-run simulation
    with open(filename, 'r') as f:
        last_line = f.readlines()[-1]
        if "Error Object: #f" in last_line:
            tui.solve.set.under_relaxation("pressure",0.3,"density",1,"body-force",1,"mom",0.7,"mp",0.5)
            tui.solve.initialize.initialize_flow()
            tui.solve.dual_time_iterate(600000,50,"no","yes")
    f.close()

    # Contour object has to be created in Case file, this will simply save the image generated at end of fill
    tui.display.objects.display("fill-contour")
    filename="Fill_Contour"+str(count)
    tui.display.save_picture(filename)

    # Target function (Resin Vf) is stored in the last line of the report file, configured in Case file itself. 
    with open('Output.txt', 'r') as f:
        last_line = f.readlines()[-1]    
        outputvf = float((last_line.split())[1]) ## Output Variable
        f.close()

    count = count + 1

    ## Force quits simulation due to FLUENT Not Responding error 
    tui.exit("yes")
    os.system("taskkill /f /im cx2220.exe")
    
    return outputvf

In [8]:
## Change parameter sample space here

optimizer = BayesianOptimization(f=None, pbounds={'lmbda':[1e-4,1e5],'power':[1e-4,1],'eta_0':[1e-2,1e1],
                                                  'eta_inf':[1e-3,1e1]}, verbose=2, random_state=1234)

In [9]:
## Change value of kappa here for optimization, xi plays no role

utility = UtilityFunction(kind='ucb',kappa=10000,xi=10000)

In [5]:
## Function to add previously obtained simulation data .txt file to optimizer before next set of iterations.

def AcquireValues(filename):
    with open(filename,'r') as f:
        dat = f.read()
    dat = dat.split('\n')
    dat = dat[1:-1]
    for i,j in enumerate(dat):
        dat[i] = [float(x) for x in j.split('\t')]
    return dat

## Combine all Parameters_Archive.txt files from previous simulations into a file called Parameters_Archive_Prior.txt

data = AcquireValues('Parameters_Archive_Prior.txt')
key_set = ['lmbda','power','eta_0','eta_inf']
for i in data:
    next_values = {}
    for key_idx,key in enumerate(key_set):
        next_values[key] = i[key_idx]
    target = i[4]
    optimizer.register(params = next_values,target=target)

In [ ]:
global count
count=1

## Creates .txt file showing all the parameters sampled.
with open('Parameters_Archive.txt','w') as f:
    f.writelines("lambda \t power \t eta_0 \t eta_inf \t target \n")
    f.close()
    
## CHANGE THIS VALUE TO CHANGE NUMBER OF ITERATIONS RUN!!!!!!    
while count < 501:
    next_values = optimizer.suggest(utility)
    with open('Parameters_Archive.txt','a') as f:
        par = str(next_values['lmbda']) + '\t' + str(next_values['power']) + '\t' + str(next_values['eta_0']) + '\t' + str(next_values['eta_inf']) + '\t'
        f.writelines(par)
        f.close()
    
    target = run_ansys_sim(**next_values)
    
    with open('Parameters_Archive.txt','a') as f:
        f.writelines(str(target) + '\n')
        f.close()
    
    try:
        optimizer.register(params = next_values,target=target)
    except:
        print('Got some errors')

In [ ]:
## Prints the best results from the entire optimizer registry as a float number

print("Best Results: {}; Airvf: {:.6f}".format(optimizer.max['target'],optimizer.max['target']))